In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that performs sparse matrix-vector multiplication.
  Given a sparse matrix $A$ of dimensions $M \times N$ and a dense vector $x$ of length $N$,
  compute the product vector $y = A \times x$, which will have length $M$. <code>A</code> is stored in row-major order.
  <code>nnz</code> is the number of non-zero elements in <code>A</code>.
</p>

<p>
  Mathematically, the operation is defined as:
  $$
  y_i = \sum_{j=0}^{N-1} A_{ij} \cdot x_j \quad \text{for} \quad i = 0, 1, \ldots, M-1
  $$
</p>

<p>
  The matrix $A$ is approximately 60 - 70% sparse.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only GPU native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in vector <code>y</code></li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
Matrix $A$ ($3 \times 4$):
$$
\begin{bmatrix}
5.0 & 0.0 & 0.0 & 1.0 \\
0.0 & 2.0 & 3.0 & 0.0 \\
0.0 & 0.0 & 0.0 & 4.0
\end{bmatrix}
$$
Vector $x$:
$$
\begin{bmatrix}
1.0 \\
2.0 \\
3.0 \\
4.0
\end{bmatrix}
$$
Output:<br>
Vector $y$:
$$
\begin{bmatrix}
9.0 \\
13.0 \\
16.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 10,000</li>
  <li>The matrix $A$ is approximately 60-70% sparse (i.e., 60-70% of elements are zero)</li>

  <li>Performance is measured with <code>M</code> = 1,000, <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, x, y are device pointers
extern "C" void solve(const float* A, const float* x, float* y, int M, int N, int nnz) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, x, y are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor, x: cute.Tensor, y: cute.Tensor, M: cute.Int32, N: cute.Int32, nnz: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, x are tensors on the GPU
@jax.jit
def solve(A: jax.Array, x: jax.Array, M: int, N: int, nnz: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    x: UnsafePointer[Float32, MutExternalOrigin],
    y: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    nnz: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, x, y are tensors on the GPU
def solve(A: torch.Tensor, x: torch.Tensor, y: torch.Tensor, M: int, N: int, nnz: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# A, x, y are tensors on the GPU
def solve(A: torch.Tensor, x: torch.Tensor, y: torch.Tensor, M: int, N: int, nnz: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Sparse Matrix-Vector Multiplication",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self, A: torch.Tensor, x: torch.Tensor, y: torch.Tensor, M: int, N: int, nnz: int
    ):
        # Accept A as either flattened (M*N,) or 2D (M, N)
        if A.shape == (M * N,):
            A_matrix = A.view(M, N)
        elif A.shape == (M, N):
            A_matrix = A
        else:
            raise AssertionError(f"A.shape {A.shape} does not match expected {(M*N,)} or {(M, N)}")
        assert x.shape == (N,)
        assert y.shape == (M,)
        result = torch.matmul(A_matrix, x)
        y.copy_(result)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "x": (ctypes.POINTER(ctypes.c_float), "in"),
            "y": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "nnz": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        A = torch.tensor(
            [5.0, 0.0, 0.0, 1.0, 0.0, 2.0, 3.0, 0.0, 0.0, 0.0, 0.0, 4.0], device="cuda", dtype=dtype
        )
        x = torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype)
        y = torch.empty(3, device="cuda", dtype=dtype)
        return {
            "A": A,
            "x": x,
            "y": y,
            "M": 3,
            "N": 4,
            "nnz": 5,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []
        # small_test
        tests.append(
            {
                "A": torch.tensor([[1.0, 2.0], [3.0, 4.0]], device="cuda", dtype=dtype),
                "x": torch.tensor([1.0, 1.0], device="cuda", dtype=dtype),
                "y": torch.empty(2, device="cuda", dtype=dtype),
                "M": 2,
                "N": 2,
                "nnz": 4,
            }
        )
        # identity_test
        tests.append(
            {
                "A": torch.tensor(
                    [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], device="cuda", dtype=dtype
                ),
                "x": torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=dtype),
                "y": torch.empty(3, device="cuda", dtype=dtype),
                "M": 3,
                "N": 3,
                "nnz": 3,
            }
        )
        # zero_test
        tests.append(
            {
                "A": torch.zeros((2, 3), device="cuda", dtype=dtype),
                "x": torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=dtype),
                "y": torch.empty(2, device="cuda", dtype=dtype),
                "M": 2,
                "N": 3,
                "nnz": 0,
            }
        )
        # single_element_per_row
        tests.append(
            {
                "A": torch.tensor(
                    [[1.0, 0.0, 0.0, 0.0], [0.0, 2.0, 0.0, 0.0], [0.0, 0.0, 3.0, 0.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "x": torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype),
                "y": torch.empty(3, device="cuda", dtype=dtype),
                "M": 3,
                "N": 4,
                "nnz": 3,
            }
        )
        # negative_values
        tests.append(
            {
                "A": torch.tensor(
                    [[-1.0, -2.0, -3.0], [-4.0, -5.0, -6.0]], device="cuda", dtype=dtype
                ),
                "x": torch.tensor([-1.0, -2.0, -3.0], device="cuda", dtype=dtype),
                "y": torch.empty(2, device="cuda", dtype=dtype),
                "M": 2,
                "N": 3,
                "nnz": 6,
            }
        )
        # medium_matrix
        tests.append(
            {
                "A": torch.tensor(
                    [
                        1.0,
                        0.0,
                        2.0,
                        0.0,
                        0.0,
                        3.0,
                        0.0,
                        4.0,
                        0.0,
                        5.0,
                        0.0,
                        6.0,
                        0.0,
                        0.0,
                        7.0,
                        0.0,
                        8.0,
                        0.0,
                        9.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                        2.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        3.0,
                        0.0,
                        0.0,
                        4.0,
                        5.0,
                        0.0,
                        0.0,
                        6.0,
                        0.0,
                        0.0,
                        7.0,
                        0.0,
                        8.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        9.0,
                        0.0,
                        1.0,
                        0.0,
                        2.0,
                        0.0,
                        3.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                        4.0,
                        5.0,
                        6.0,
                        0.0,
                        7.0,
                        8.0,
                        0.0,
                        0.0,
                        0.0,
                        9.0,
                        0.0,
                        1.0,
                        0.0,
                        2.0,
                        3.0,
                        0.0,
                        0.0,
                        0.0,
                        4.0,
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "x": torch.tensor(
                    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], device="cuda", dtype=dtype
                ),
                "y": torch.empty(10, device="cuda", dtype=dtype),
                "M": 10,
                "N": 8,
                "nnz": 35,
            }
        )

        # random_sparse_matrix
        M_sparse = 20
        N_sparse = 20
        sparsity = 0.65

        # Generate random sparse matrix
        A_dense = torch.empty((M_sparse, N_sparse), device="cuda", dtype=dtype).uniform_(-5.0, 5.0)
        mask = torch.rand((M_sparse, N_sparse), device="cuda") > sparsity
        A_sparse = A_dense * mask
        nnz_sparse = int(mask.sum().item())

        tests.append(
            {
                "A": A_sparse,
                "x": torch.empty(N_sparse, device="cuda", dtype=dtype).uniform_(-2.0, 2.0),
                "y": torch.zeros(M_sparse, device="cuda", dtype=dtype),
                "M": M_sparse,
                "N": N_sparse,
                "nnz": nnz_sparse,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M = 1000
        N = 10000
        nnz = 3500000
        A = torch.zeros((M, N), device="cuda", dtype=dtype)
        total_elements = M * N
        flat_indices = torch.randperm(total_elements, device="cuda")[:nnz]
        values = torch.empty(nnz, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        A.view(-1)[flat_indices] = values

        # Create a mask: 35% entries will be kept, 65% set to zero
        x = torch.empty(N, device="cuda", dtype=dtype).uniform_(-5.0, 5.0)
        y = torch.empty(M, device="cuda", dtype=dtype)
        return {
            "A": A,
            "x": x,
            "y": y,
            "M": M,
            "N": N,
            "nnz": nnz,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
